In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd
import tensorflow as tf
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Add path to SHARE to import ONNX wrappers
share_path = os.path.abspath("../SHARE")
if share_path not in sys.path:
    sys.path.append(share_path)

# Try importing ONNX modules
try:
    from ONNX_Predict.LSTM_onnx import LSTM_onnx 
    from ONNX_Predict.Scaler_onnx import Scaler_onnx
    print("ONNX modules imported successfully.")
except ImportError as e:
    print(f"Error importing ONNX modules: {e}")
    print("Make sure ONNX_Predict package is available (e.g. check ../SHARE/).")

# Import original transition function setup from local directory
from transition_function_model import setup_transition_function_model

class TransitionFunctionModelONNX:
    def __init__(self, ice_folder, pg_folder, soc_ini=0.7):
        """
        Equivalent to transition_function_model but using ONNX models.
        """
        self.ice_folder = ice_folder
        self.pg_folder = pg_folder
        
        # Load Scalers (using ONNX_Predict.Scaler_onnx)
        self.ice_scaler_in = Scaler_onnx('scaler_input.onnx', ice_folder)
        self.ice_scaler_out = Scaler_onnx('scaler_output.onnx', ice_folder)
        self.ice_scaler_inv_out = Scaler_onnx('scaler_inverse_output.onnx', ice_folder)
        
        self.pg_scaler_in = Scaler_onnx('scaler_input.onnx', pg_folder)
        self.pg_scaler_out = Scaler_onnx('scaler_output.onnx', pg_folder)
        self.pg_scaler_inv_out = Scaler_onnx('scaler_inverse_output.onnx', pg_folder)

        # Load Models (tf_model arg is passed but may be unused/dummy depending on impl)
        self.ice_model = LSTM_onnx('ICE_onnx.onnx', ice_folder, 'model.h5')
        self.pg_model = LSTM_onnx('PG_onnx.onnx', pg_folder, 'model.h5')
        
        # Initial State
        self.soc_ini = soc_ini
        self.reset_states()

    def reset_states(self):
        # Reset internal model states
        self.ice_model.reset_states()
        self.pg_model.reset_states()
        
        # Initialize Aux vectors (feed-back inputs)
        # ICE: [Torque, NO, NO2, CO, CO2] -> All 0
        y_ini_ice_raw = np.zeros((1, 5), dtype='float32') 
        self.ice_aux = self.ice_scaler_out.transform(y_ini_ice_raw)[0].reshape(1, 1, 5)
        
        # PG: [Velocity, SOC] -> [0, soc_ini]
        y_ini_pg_raw = np.array([[0.0, self.soc_ini]], dtype='float32')
        self.pg_aux = self.pg_scaler_out.transform(y_ini_pg_raw)[0].reshape(1, 1, 2)
        
    def predict_ice(self, speed_rpm, m_fuel_mg, t_amb_k, p_amb_bar):
        """
        Predict ICE outputs. Inputs are scalars or single values.
        Returns physical units: Torque, NO, NO2, CO, CO2
        """
        # 1. Prepare input vector (1, 4)
        x = np.array([speed_rpm, m_fuel_mg, t_amb_k, p_amb_bar], dtype='float32').reshape(1, 4)
        
        # 2. Scale
        x_scaled = self.ice_scaler_in.transform(x).reshape(1, 1, 4)
        
        # 3. Predict
        # Pass x_scaled and the initial aux state
        y_scaled = self.ice_model([x_scaled, self.ice_aux])
        
        # 4. Inverse Scale
        # y_scaled shape is likely (1, 1, 5), need (1, 5) for scaler
        y_scaled_flat = y_scaled.reshape(1, 5)
        y = self.ice_scaler_inv_out.transform(y_scaled_flat)
        
        # Return unpacked values
        return y[0]

    def predict_pg(self, ice_speed_soll, em2_torque, ice_torque, brake_perc):
        """
        Predict PG outputs.
        Returns physical units: Car_Speed, SOC
        """
        # 1. Prepare input (1, 4)
        x = np.array([ice_speed_soll, em2_torque, ice_torque, brake_perc], dtype='float32').reshape(1, 4)
        
        # 2. Scale
        x_scaled = self.pg_scaler_in.transform(x).reshape(1, 1, 4)
        
        # 3. Predict
        y_scaled = self.pg_model([x_scaled, self.pg_aux])
        
        # 4. Inverse Scale
        y_scaled_flat = y_scaled.reshape(1, 2)
        y = self.pg_scaler_inv_out.transform(y_scaled_flat)
        
        return y[0]

def setup_transition_function_model_onnx(ice_path, pg_path, soc_ini=0.7):
    return TransitionFunctionModelONNX(ice_path, pg_path, soc_ini)

In [ ]:
def run_experiment_1(trans_func, mode='tf', steps=10000):
    """
    Case 1: Idling at 1000 rpm with 3 mg of fuel.
    Predicts NOx.
    """
    data = {"step": [], "nox": []}
    
    # Inputs
    rpm, fuel = 1000.0, 3.0
    temp, press = 298.0, 1.0
    
    # TF Constants
    if mode == 'tf':
        rpm_t = tf.constant(rpm, dtype=tf.float32)
        fuel_t = tf.constant(fuel, dtype=tf.float32)
        temp_t = tf.constant(temp, dtype=tf.float32)
        press_t = tf.constant(press, dtype=tf.float32)
    
    trans_func.reset_models() # Ensure clean state
    start_time = time.time()
    
    for k in range(steps):
        if mode == 'tf':
            res = trans_func.predict_ice(rpm_t, fuel_t, temp_t, press_t)
            nox_val = float(res[1].numpy())
        else:
            res = trans_func.predict_ice(rpm, fuel, temp, press)
            nox_val = float(res[1])
            
        data["step"].append(k)
        data["nox"].append(nox_val)
        
    end_time = time.time()
    return pd.DataFrame(data), end_time - start_time

def run_experiment_2(trans_func, mode='tf', total_steps=5000, interval=400):
    """
    Case 2: 2500 rpm, variable fuel mass. 
    Predicts Torque.
    """
    fuelmass_profile = [3, 10, 20, 30, 40, 50, 60, 70, 5]
    data = {"step": [], "mf": [], "torque": []}
    
    rpm = 2500.0
    temp, press = 298.0, 1.0
    
    if mode == 'tf':
        rpm_t = tf.constant(rpm, dtype=tf.float32)
        temp_t = tf.constant(temp, dtype=tf.float32)
        press_t = tf.constant(press, dtype=tf.float32)

    trans_func.reset_models()
    start_time = time.time()

    for step in range(total_steps):
        idx = step // interval
        mf_val = fuelmass_profile[idx] if idx < len(fuelmass_profile) else fuelmass_profile[-1]
        
        if mode == 'tf':
            mf_t = tf.constant(mf_val, dtype=tf.float32)
            res = trans_func.predict_ice(rpm_t, mf_t, temp_t, press_t)
            torque_val = float(res[0].numpy())
        else:
            res = trans_func.predict_ice(rpm, mf_val, temp, press)
            torque_val = float(res[0])

        data["step"].append(step)
        data["mf"].append(mf_val)
        data["torque"].append(torque_val)
        
    end_time = time.time()
    return pd.DataFrame(data), end_time - start_time

def run_experiment_3(trans_func, mode='tf', total_steps=5000, interval=800):
    """
    Case 3: ICE Stopped, Variable EM2 Torque.
    Predicts SOC and Velocity.
    """
    em2_torque_profile = [50, 250, 400]
    data = {"step": [], "em2_torque": [], "soc": [], "vel": []}
    
    ice_speed, fuel, brake = 0.0, 0.0, 0.0
    temp, press = 298.0, 1.0
    
    if mode == 'tf':
        ice_sp_t = tf.constant(ice_speed, dtype=tf.float32)
        fuel_t = tf.constant(fuel, dtype=tf.float32)
        brake_t = tf.constant(brake, dtype=tf.float32)
        temp_t = tf.constant(temp, dtype=tf.float32)
        press_t = tf.constant(press, dtype=tf.float32)

    trans_func.reset_models() # Ensure clean state (and soc_ini)
    start_time = time.time()

    for step in range(total_steps):
        idx = step // interval
        em2_val = em2_torque_profile[idx] if idx < len(em2_torque_profile) else em2_torque_profile[-1]
        
        if mode == 'tf':
            em2_t = tf.constant(em2_val, dtype=tf.float32)
            # ICE Prediction (gives torque needed for PG)
            ice_res = trans_func.predict_ice(ice_sp_t, fuel_t, temp_t, press_t)
            ice_torque = ice_res[0]
            
            # PG Prediction
            pg_res = trans_func.predict_PG(ice_sp_t, em2_t, ice_torque, brake_t)
            vel_val = float(pg_res[0].numpy())
            soc_val = float(pg_res[1].numpy())
        else:
            # ONNX
            ice_res = trans_func.predict_ice(ice_speed, fuel, temp, press)
            ice_torque = ice_res[0]
            
            pg_res = trans_func.predict_pg(ice_speed, em2_val, ice_torque, brake)
            vel_val = float(pg_res[0])
            soc_val = float(pg_res[1])

        data["step"].append(step)
        data["em2_torque"].append(em2_val)
        data["soc"].append(soc_val)
        data["vel"].append(vel_val)

    end_time = time.time()
    return pd.DataFrame(data), end_time - start_time

In [ ]:
def run_comparison(name, runner_fn, ice_tf, pg_tf, ice_onnx, pg_onnx, **runner_kwargs):
    print(f"\n{'='*20} Running {name} {'='*20}")
    
    # 1. TF Experiment
    print("> Starting TensorFlow Experiment...")
    # Handle SOC_ini for Case 3 which uses 0.5 instead of default 0.7
    soc_ini = 0.5 if name == "Case3" else 0.7
    
    tf_model = setup_transition_function_model(ice_tf, pg_tf, SOC_ini=soc_ini)
        
    df_tf, time_tf = runner_fn(tf_model, mode='tf', **runner_kwargs)
    print(f"  TF Done. Time: {time_tf:.4f} s")
    
    # 2. ONNX Experiment
    print("> Starting ONNX Experiment...")
    onnx_model = setup_transition_function_model_onnx(ice_onnx, pg_onnx, soc_ini=soc_ini)
    df_onnx, time_onnx = runner_fn(onnx_model, mode='onnx', **runner_kwargs)
    print(f"  ONNX Done. Time: {time_onnx:.4f} s")
    
    # 3. Calculate Metrics
    print("> Calculating Metrics...")
    metrics_list = []
    
    # Identify value columns (exclude step, etc)
    skip_cols = ['step', 'mf', 'em2_torque']
    val_cols = [c for c in df_tf.columns if c not in skip_cols and c in df_onnx.columns]
    
    for col in val_cols:
        tf_vals = df_tf[col].values
        onnx_vals = df_onnx[col].values
        
        diff = np.abs(tf_vals - onnx_vals)
        mae = np.mean(diff)
        rmse = np.sqrt(np.mean(diff**2))
        max_d = np.max(diff)
        
        metrics_list.append({
            "Variable": col,
            "MAE": mae,
            "RMSE": rmse,
            "Max Diff": max_d
        })
    
    metrics_df = pd.DataFrame(metrics_list)
    print("\nPerformance Metrics:")
    print(metrics_df)
    
    # Avoid division by zero
    speedup = time_tf / time_onnx if time_onnx > 1e-6 else 0
    print(f"\nSpeedup (TF/ONNX): {speedup:.2f}x")
    
    # 4. Generate Plot
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1,
                        subplot_titles=(f"{name} Values", f"{name} Absolute Difference"))
    
    # Assuming 0.5s step as per main.ipynb comments
    time_axis = df_tf['step'] / 2 
    
    color_cycle = px.colors.qualitative.Plotly
    
    for i, col in enumerate(val_cols):
        color = color_cycle[i % len(color_cycle)]
        # TF line
        fig.add_trace(go.Scatter(x=time_axis, y=df_tf[col], name=f"{col} (TF)",
                                 line=dict(dash='solid', color=color), legendgroup=col), row=1, col=1)
        # ONNX line
        fig.add_trace(go.Scatter(x=time_axis, y=df_onnx[col], name=f"{col} (ONNX)",
                                 line=dict(dash='dot', color=color), legendgroup=col, showlegend=False), row=1, col=1)
        
        # Diff line
        diff = np.abs(df_tf[col] - df_onnx[col])
        fig.add_trace(go.Scatter(x=time_axis, y=diff, name=f"Diff {col}",
                                 line=dict(color=color), legendgroup=col, showlegend=False), row=2, col=1)

    fig.update_layout(height=800, title_text=f"{name}: TF vs ONNX Comparison<br>Speedup: {speedup:.2f}x")
    fig.update_xaxes(title_text="Time [s]", row=2, col=1)
    
    # Save Plot
    output_dir = "Model_Discussion_Comparison"
    os.makedirs(output_dir, exist_ok=True)
    html_file = os.path.join(output_dir, f"{name}_comparison.html")
    fig.write_html(html_file)
    print(f"Saved plot to {html_file}")
    
    return metrics_df

# Define Paths
# Relative to 'controller_for_ICE_PG/model_discussion/'
# TF Models in src (../src/...)
tf_ice_path = "../src/models_markus/ICE_Model_Update_01"
tf_pg_path = "../src/models_markus/PG_v2"

# ONNX Models in SHARE (../SHARE/CTTC_models/ONNX/...)
onnx_ice_path = "../SHARE/CTTC_models/ONNX/ICE"
onnx_pg_path = "../SHARE/CTTC_models/ONNX/PG"

# Execute Experiments
print(f"Current Working Directory: {os.getcwd()}")
print("Checking paths...")
print(f"TF ICE Exists: {os.path.exists(tf_ice_path)}")
print(f"ONNX ICE Exists: {os.path.exists(onnx_ice_path)}")

# Case 1
run_comparison("Case1", run_experiment_1, tf_ice_path, tf_pg_path, onnx_ice_path, onnx_pg_path, steps=5000*2)

# Case 2
run_comparison("Case2", run_experiment_2, tf_ice_path, tf_pg_path, onnx_ice_path, onnx_pg_path, total_steps=2500*2)

# Case 3
run_comparison("Case3", run_experiment_3, tf_ice_path, tf_pg_path, onnx_ice_path, onnx_pg_path, total_steps=2500*2)